This notebook fits by-cre models on the shendure data & demonstrates that estimated nb parameters track closely with UMI means, as expected.

The matrix-creation and fitting code is independent of the main scMPRA package, since its this analysis that informed the package code.

# Setup

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
#imports
import pandas as pd
import numpy as np
import time
import pickle
from formulaic import Formula
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import linregress
from tensorzinb.tensorzinb import TensorZINB
import scMPRAforge as scm

import tqdm

In [3]:
#create dask cluster
from dask.distributed import Client, LocalCluster
cluster=LocalCluster(memory_limit='8GB')
client = Client(cluster)

# Describe with an ortho

In [4]:
#load data
data_root="/gpfs/gibbs/pi/reilly/tabula_data"
shendure=scm.scMPRA_data.from_tsv(f"{data_root}/shendure/shendure_counts_grouped.txt")
shendure.set_negative_controls(["minP","noP"])
shendure.set_reference_cell("Pluripotent")
shendure.ortho_filter()

scMPRAforge: INFO: Dropped 372 of 1833 (cell_type, cre_id) combos with fewer than 3 nonzero entries.


In [12]:
path="/gpfs/gibbs/pi/reilly/tabula_data/shendure"
name="ortho_primordial"

import os
if os.path.isdir(path+"/"+name):
    print("[+] Model found. Loading...")
    primordial=scm.ortho.load(client,path,name)
else:
    print("[+] Model not found. Creating...")
    primordial=scm.ortho()
    primordial.criss_cross(client=client,
                       dat=shendure)
    primordial.extract_params(client)
    primordial.save(path,name)


[+] Model found. Loading...


In [13]:
primordial

In [ ]:
#primordial.load

In [ ]:
primordial.by_cell_type.model

In [ ]:
primordial.by_cell_type

In [ ]:
scm.nb_versus_means()

In [ ]:
x.result()

In [ ]:
primordial.by_cell_type.split

In [ ]:
primordial.by_cre_parameters.result()

In [ ]:
primordial.by_cell_type_parameters

In [ ]:
try:
    result = primordial.by_cell_type_parameters.result()
except Exception as e:
    import traceback
    traceback.print_exc()  # or print(str(e))



In [ ]:
QC_cre=scm.nb_versus_means(params=cre_params,
                design_matricies=cre_mats,
                scMPRAdat=shendure)

QC_ct=scm.nb_versus_means(params=ct_params,
                design_matricies=ct_mats,
                scMPRAdat=shendure)

# Examine QC metrics

TODO: check the one that failed.

Now that we've generated some QC metrics, let's examine them. First, let's look at the mu min & max : none should be below zero, none should be above 1000.

In [ ]:
def minimax(QC):
    x=[]
    for level in QC.keys():
        if QC[level]["success"]:
            x.append(QC[level]["dat"])
    x=pd.concat(x)
    print(f"min {min(x['mu'])}, max {max(x['mu'])}")

print("cre")
minimax(QC_cre)
print("ct")
minimax(QC_ct)

All in the right ballpark!

Now let's look at the correlations.

In [ ]:
r=[]
for QC in [QC_cre,QC_ct]:
    print("---")
    
    for level in QC.keys():
        if QC[level]["success"]:
            if np.isnan(QC[level]["r_value"]):
                print(f"nan in {level}")
            else:
                if QC[level]["r_value"]<0.8:
                    print(f"low r {level}")
                r.append(QC[level]["r_value"])


sns.violinplot(r)

print(r)
#print(np.mean(r))
#for cell_type in QC:
#    print(f"{cell_type} : r={QC[cell_type]['r_value']}, slope={QC[cell_type]['slope']}")

Correlations generally look pretty good. Let's examine the cases where they aren't.

Notably all of the bad ones are from the set of "by cell-type" models.

I'm going to guess these are low-expressing CREs. Let's take a look.

In [ ]:
highlighted_cre_ids=["Cdk5r1_chr11_12595","Lamb1_chr12_2206","Lamc1_chr1_12183","Txndc12_chr4_7969"]

grouped = (
    shendure.data
    .groupby(["cre_id", "cell_type"])["umis_mpra_bc"]
    .mean()
    .reset_index()
)

# Split base vs. highlighted
base = grouped[~grouped["cre_id"].isin(highlighted_cre_ids)]
highlighted = grouped[grouped["cre_id"].isin(highlighted_cre_ids)]

# Set up plot
plt.figure(figsize=(10, 6))

# Plot base layer with jitter
sns.stripplot(
    data=base,
    x="cell_type",
    y="umis_mpra_bc",
    color="lightgray",
    jitter=0.35,
    label="Other CREs",
    size=6
)

# Overlay highlighted CREs with jitter and custom color
palette = sns.color_palette("tab10", n_colors=len(highlighted_cre_ids))
for i, cre in enumerate(highlighted_cre_ids):
    sns.stripplot(
        data=highlighted[highlighted["cre_id"] == cre],
        x="cell_type",
        y="umis_mpra_bc",
        color=palette[i],
        jitter=0.35,
        label=cre,
        size=6
    )

# Y-axis and aesthetics
plt.ylim(-1, 1)
plt.xlabel("Cell Type")
plt.ylabel("Mean UMIs (mpra_bc)")
plt.title("Mean UMIs per (cre_id, cell_type)")
plt.xticks(rotation=45)
plt.legend(title="Highlighted CREs")
plt.tight_layout()
plt.show()


In [ ]:
QC_cre["Cdk5r1_chr11_12595"]

Let's examine all of them together graphically

# Graphical, CRE models

# Graphical, CT models

In [ ]:
for cell_type in QC:
    sns.regplot(x=QC[cell_type]["x"], y=QC[cell_type]["y"], scatter=True, line_kws={"color": "red"})
    plt.title(cell_type)
    plt.show()
    

In [6]:
cluster.close()